---
# **Cuaderno 3:**  Analisis de datos de ORPHANET
---
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: Agosto 2025

**Aims:**

* Analiza los datos obtenidos al filtrar por gen un archivo de mutaciones de la base de datos del ICGC

In [1]:
import pandas as pd
import requests

In [2]:
ruta_xml = "../data/Orphanet/en_product6.xml"

#Lee archivo XML de ORPHANET
xml_gen = pd.read_xml(ruta_xml, encoding="UTF-8", xpath="//Gene")
xml_disorder = pd.read_xml(ruta_xml, encoding="UTF-8", xpath=".//DisorderList/Disorder")
xml_disorder_type = pd.read_xml(ruta_xml, encoding="UTF-8", xpath=".//DisorderType")

#Agrega un prefijo para identificar el dataframe al concatenar
xml_gen = xml_gen.add_prefix("Gene_")
xml_disorder = xml_disorder.add_prefix("Disorder_")
xml_disorder_type = xml_disorder_type.add_prefix("DisorderType_")

#Concatena
df_orphanet = pd.concat([xml_gen.reset_index(drop=True),
                        xml_disorder.reset_index(drop=True), 
                        xml_disorder_type.reset_index(drop=True)], 
                        axis=1)

# Elimina columnas con \n
df_orphanet = df_orphanet.drop(columns=["Gene_SynonymList", "Gene_GeneType", "Gene_ExternalReferenceList", "Gene_LocusList", "Disorder_DisorderGroup", "Disorder_DisorderType", "Disorder_DisorderGeneAssociationList", "DisorderType_id"])
df_orphanet = df_orphanet.dropna(subset=["Gene_Symbol", "Disorder_Name"])
df_orphanet

,Gene_id,Gene_Name,Gene_Symbol,Disorder_id,Disorder_OrphaCode,Disorder_ExpertLink,Disorder_Name,DisorderType_Name
0,20160,kinesin family member 7,KIF7,17601.0,166024.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Multiple epiphyseal dysplasia-macrocephaly-fac...,Disease
1,15470,aspartylglucosaminidase,AGA,5.0,93.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Aspartylglucosaminuria,Disease
2,26792,CWC27 spliceosome associated cyclophilin,CWC27,17604.0,166035.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Brachydactyly-short stature-retinitis pigmento...,Malformation syndrome
3,15570,sulfatase modifying factor 1,SUMF1,6.0,585.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Multiple sulfatase deficiency,Disease
4,16375,mannosidase beta,MANBA,7.0,118.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Beta-mannosidosis,Disease
...,...,...,...,...,...,...,...,...
4073,18166,"ST3 beta-galactoside alpha-2,3-sialyltransfera...",ST3GAL5,31418.0,619948.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Early-onset autoimmunity-autoinflammation-immu...,Disease
4074,22522,solute carrier family 35 member A3,SLC35A3,31423.0,619979.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Developmental delay-immunodeficiency-leukoence...,Disease
4075,21881,calcium voltage-gated channel subunit alpha1 D,CACNA1D,31422.0,619972.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,CADINS disease,Disease
4076,22560,adenosine monophosphate deaminase 2,AMPD2,31415.0,619363.0,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,NOCARH syndrome,Disease


In [6]:
uniprot_ids = ['O00238']

features_dict = []

for uniprot_id in uniprot_ids:
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}"
        response = requests.get(url)
    
        if response.status_code != 200:
            print(f"Error al obtener {uniprot_id}: {response.status_code}")
            continue
    
        json_data = response.json()
        gen_name = (json_data.get("genes",[{}])[0].get("geneName",{}) or {}).get("value", uniprot_id)
        features = json_data.get("features", [])
        
        cross_references  = json_data.get("features")
        for cross in cross_references:
            if cross.get("type") == "Natural variant":
                print(cross.get("description"))

in AMD3; uncertain significance; mouse BMPR1B construct containing this mutation shows reduced GDF5-dependent receptor activation, mouse BMPR1B construct containing this mutation shows no loss of cell membrane localization; dbSNP:rs745854387
in a gastric adenocarcinoma sample; somatic mutation; dbSNP:rs200035802
in AMD3; mouse BMPR1B construct containing this mutation shows loss of GDF5-dependent receptor activation, chicken BMPR1B construct containing this mutation does not show reduced chondrocyte differentiation, mouse BMPR1B construct containing this mutation shows no loss of cell membrane localization; dbSNP:rs863225041
in dbSNP:rs34231464
in BDA2; in animal models loss of kinase activity and loss of positive regulation of chondrocyte differentiation; dbSNP:rs121434417
in dbSNP:rs35973133
in a metastatic melanoma sample; somatic mutation
in BDA1D; acts in a dominant-negative manner; dbSNP:rs869025614
in dbSNP:rs34970181
in brachydactyly type C and BDA2; with also additional featur

In [22]:
cross_references  = json_data.get("features")
for cross in cross_references:
    if cross.get("type") == "Natural variant":
        description = cross.get("description")
        description_strip = description.split(";")
        print(description_strip)

['in AMD3', ' uncertain significance', ' mouse BMPR1B construct containing this mutation shows reduced GDF5-dependent receptor activation, mouse BMPR1B construct containing this mutation shows no loss of cell membrane localization', ' dbSNP:rs745854387']
['in a gastric adenocarcinoma sample', ' somatic mutation', ' dbSNP:rs200035802']
['in AMD3', ' mouse BMPR1B construct containing this mutation shows loss of GDF5-dependent receptor activation, chicken BMPR1B construct containing this mutation does not show reduced chondrocyte differentiation, mouse BMPR1B construct containing this mutation shows no loss of cell membrane localization', ' dbSNP:rs863225041']
['in dbSNP:rs34231464']
['in BDA2', ' in animal models loss of kinase activity and loss of positive regulation of chondrocyte differentiation', ' dbSNP:rs121434417']
['in dbSNP:rs35973133']
['in a metastatic melanoma sample', ' somatic mutation']
['in BDA1D', ' acts in a dominant-negative manner', ' dbSNP:rs869025614']
['in dbSNP:rs

In [30]:
cross_references  = json_data.get("uniProtKBCrossReferences")
for cross in cross_references:
    if cross.get("database") == "MIM":
        print(cross)

{'database': 'MIM', 'id': '112600', 'properties': [{'key': 'Type', 'value': 'phenotype'}]}
{'database': 'MIM', 'id': '603248', 'properties': [{'key': 'Type', 'value': 'gene'}]}
{'database': 'MIM', 'id': '609441', 'properties': [{'key': 'Type', 'value': 'phenotype'}]}
{'database': 'MIM', 'id': '616849', 'properties': [{'key': 'Type', 'value': 'phenotype'}]}


In [28]:

for clave, valor in json_data.items():
    if "MIM" in str(valor):  # se pasa a str por si son números
        print("Valor encontrado en:", clave, "→")


Valor encontrado en: comments →
Valor encontrado en: uniProtKBCrossReferences →
